# 第33课 · 造出声音的原子 — 亲手实现正弦波（sine wave）x[n]=A·sin(2πfn/sr)

**目标**：实现 `x[n] = A·sin(2π·f·n/sr)`，手动写出每一步，再用 `np.allclose` 验证与 `aurora.audio.sine` 逐点一致。

`aurora.audio.sine` 是后续 STFT 和 mel 特征提取的测试信号来源，`my_sine` 实现的正是同一套公式。

← **上一课**　[L32 · NumPy 信号基础](L32_numpy_signals.ipynb)

> 上节课学习了 **NumPy 信号基础**：np.arange / linspace，生成 16kHz 一秒时间轴。  
> 本课将探讨 **正弦波生成**。

## 本课剧情：正弦波是声音的原子

物理学有个惊人结论：**任何声音都能分解为若干正弦波之和**（这就是傅里叶定理）。  
钢琴中央 C（261 Hz）、吉他弦声、人的嗓音——本质上都是不同频率、不同振幅的正弦波叠加。

正弦波的公式：

$$x[n] = A \cdot \sin\!\left(2\pi f \cdot \frac{n}{f_s}\right)$$

读法：在第 `n` 个采样点，圆圈已经转了 `2πf·n/fₛ` 弧度，`sin` 取这个角度的投影，乘以振幅 `A`。

| 变量 | 含义 | 典型值 |
|---|---|---|
| `f` | 频率（Hz），决定音调高低 | 440（钢琴 A4） |
| `fₛ` | 采样率（Hz），决定时间分辨率 | 16000 或 44100 |
| `A` | 振幅，决定响度 | 1.0（归一化） |
| `n` | 采样点编号 = `np.arange(N)` | 0, 1, 2, ..., N-1 |

本课任务：实现 `my_sine(freq, duration, sample_rate)` 并与 `aurora.audio.sine`（Aurora 的生产实现）逐点对比。

## 1. 导入工具 + 仓库的"标准答案"

这一段是为了让你一边写自己的版本，一边随时拿仓库版本对照。学习时最怕“只看答案，不知道答案为什么长这样”。所以我们会先把参考实现摆在旁边，再逐步拆公式。


## 实验入口：把声音拆成可观察的数组

用 `sample_rate=8`、`duration=1.0`、`freq=2.0` 生成 8 个采样点，每一步的中间量都打印出来，可以直接看到 `n`、`t`、`angle`、`wave` 的数值对应关系。

In [ ]:
import numpy as np
from aurora.audio import sine as reference_sine  # 仓库实现 = 你的对照答案
print('就绪')

## 动手观察：序列怎样一步步变成信号

观察 `n`、`t`、`angle` 三个中间数组：`angle` 是 `t` 乘以 `2π·freq`，当 `freq=2` 时，1 秒内 `angle` 从 0 走到 4π，对应两个完整周期。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = int(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：遍历频率、振幅和相位

`freq` 加倍时，相同时间段内的过零点数量也加倍；`amplitude` 只改变数值范围，不影响波形形状。

In [ ]:
import numpy as np

sample_rate = 16
duration = 0.5
t = np.arange(int(duration * sample_rate)) / sample_rate

for freq in [1, 2, 4]:
    wave = np.sin(2 * np.pi * freq * t)
    print(f'freq={freq}Hz ->', np.round(wave, 3))

for amplitude in [0.5, 1.0, 2.0]:
    wave = amplitude * np.sin(2 * np.pi * 2 * t)
    print(f'amplitude={amplitude} -> min={wave.min():.1f}, max={wave.max():.1f}')


## 2. 手算一个采样点

频率 `f=1 Hz`，采样率 `fₛ=8 Hz`，振幅 `A=1`。

**第 0 点** (`n=0`)：角度 = 2π·1·0/8 = 0，x[0] = sin(0) = **0**  
**第 2 点** (`n=2`)：角度 = 2π·1·2/8 = π/2，x[2] = sin(π/2) = **1**（峰值）  
**第 4 点** (`n=4`)：角度 = 2π·1·4/8 = π，x[4] = sin(π) = **0**（过零）  
**第 6 点** (`n=6`)：角度 = 2π·1·6/8 = 3π/2，x[6] = sin(3π/2) = **-1**（谷值）

8 个点恰好完成一个完整周期——因为 `fₛ/f = 8/1 = 8`（奈奎斯特定理：完整一圈需要至少 2 个点）。

> **Aurora 连接**：`aurora.audio.sine` 正是这个公式；它是所有合成音频测试的基础信号，也是 FFT 验证套件（`tests/audio/test_transforms.py`）中"单频纯音"测试用例的信号源。

## 3. ✏️ 你的任务：实现 `my_sine`

这一题最关键的不是公式本身，而是把公式翻译成代码时，知道每个变量在代码里扮演什么角色：

- `freq`：频率，决定波动快慢
- `duration`：时长，决定序列有多长
- `sample_rate`：采样率（sample rate，sr），决定每秒取多少点
- `n` 或 `t`：位置序列，代表每个采样点的编号或时间
- `amplitude`：振幅，决定波形上下摆动的幅度

**推理路线**：
1. 先算出总点数：`N = int(duration * sample_rate)`（截断取整，与仓库 `aurora.audio.sine` 一致），这是返回数组的长度。
2. 生成整数编号序列：`n = np.arange(N)`，从 0 到 N-1，不含端点 N。
3. 把编号换算成角度：`angle = 2*np.pi*freq*n/sample_rate`，其中 `n/sample_rate` 就是每个点的时间 `t`。
4. 返回 `amplitude * np.sin(angle)`——振幅只是一个缩放因子，不影响波形形状。

**参考输入输出**：`freq=1, duration=1, sample_rate=8` → 8 个点，`angle=[0, π/4, π/2, 3π/4, π, 5π/4, 3π/2, 7π/4]`，`sin≈[0, 0.707, 1, 0.707, 0, -0.707, -1, -0.707]`

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。



### 写 `my_sine` 前明确三件事

- 输入：`freq`（Hz）、`duration`（秒）、`sample_rate`（点/秒）、`amplitude`（默认 1.0）
- 关键步骤：`N = int(duration * sample_rate)`，`n = np.arange(N)`，`angle = 2*np.pi*freq*n/sample_rate`
- 返回：长度为 `N` 的 float64 数组，值域 `[-amplitude, amplitude]`

In [ ]:
def my_sine(freq, duration, sample_rate, amplitude=1.0):
    # ✏️ TODO: 按公式实现并返回正弦波
    raise NotImplementedError("TODO: 按公式实现并返回正弦波")


## 4. 对答案：和仓库实现逐点比较

这里要看的不只是“有没有报错”，还要看三个层面的对应关系：

- **长度是否一致**：说明你有没有生成正确数量的采样点
- **每个点是否一致**：说明你有没有把公式写对
- **误差是否极小**：说明数值计算里的浮点误差是否在正常范围

如果这里没过，不要急着改 `sin` 本身，先检查 `n` 是不是从 0 开始、`N` 算得对不对、采样率是不是除错了。


In [ ]:
# ⚡ 形状 + 数值验证（未实现时给出提示而非崩溃）
try:
    freq, dur, sr = 440.0, 1.0, 16000
    mine = my_sine(freq, dur, sr)
    ref = reference_sine(freq, duration=dur, sample_rate=sr)
    max_diff = float(np.max(np.abs(mine - ref)))
    print('我的 shape:', mine.shape, '| 标准 shape:', ref.shape)
    print('最大逐点误差:', f'{max_diff:.2e}')
    assert mine.shape == ref.shape, '长度应一致'
    assert max_diff < 1e-12, f'数值应几乎一致（机器精度级别），当前误差={max_diff:.2e}'
    print('\n✅ 对答案通过：你的正弦波和仓库一致。')
except NotImplementedError:
    print('⚠️  my_sine 尚未实现，请先完成 cell 上方的 TODO，再运行此验证。')
except TypeError as e:
    print(f'❌ 类型错误（my_sine 可能返回了 None）：{e}')


In [ ]:
# ⚡ 边缘测试：覆盖 int() 截断和 amplitude=0 情况
try:
    # 边缘 1：amplitude=0 应返回全零数组
    result_zero = my_sine(freq=440.0, duration=1.0, sample_rate=16000, amplitude=0.0)
    assert np.all(result_zero == 0.0), '❌ amplitude=0 时应返回全零数组'
    print('✅ 边缘1 通过：amplitude=0 返回全零数组')

    # 边缘 2：int() 截断（duration=0.5, sr=3 → int(1.5)=1，round(1.5)=2）
    result_trunc = my_sine(freq=1.0, duration=0.5, sample_rate=3)
    ref_trunc = reference_sine(freq=1.0, duration=0.5, sample_rate=3)
    assert result_trunc.shape == ref_trunc.shape, (
        f'❌ int() 截断测试：shape 不一致，my={result_trunc.shape} vs ref={ref_trunc.shape}\n'
        '   提示：确保用 int(duration * sample_rate) 而非 round()'
    )
    print(f'✅ 边缘2 通过：int截断 duration=0.5,sr=3 → N={result_trunc.shape[0]}（与仓库一致）')
except NotImplementedError:
    print('⚠️  my_sine 尚未实现，完成后再运行边缘测试。')


## 5. 画出来看看（前 50 个采样点）

图像能直接显示序列的时间变化。你看到的不是“数组”，而是“这个波在时间上怎么上下摆动”。

前 50 个采样点只是一个很小的窗口，但它能帮你看出：
- 波形是不是连续的
- 振幅是不是对的
- 频率是不是符合预期

如果你愿意，还可以把 `mine[:50]` 和 `ref[:50]` 叠在一起看，这样更容易发现偏差。


In [ ]:
import matplotlib.pyplot as plt
try:
    mine
except NameError:
    print('⚠️  my_sine 尚未实现，跳过绘图。完成上方 TODO 后重新运行此格。')
else:
    plt.figure(figsize=(8, 3))
    plt.plot(mine[:50], marker='o', ms=3)
    plt.title('440 Hz sine — first 50 samples')
    plt.xlabel('sample n'); plt.ylabel('amplitude')
    plt.tight_layout(); plt.show()

**🎉 完成后**：`git commit -m 'learn: L33 my_sine'`

## 🎨 图示：440Hz 正弦的前 50 个采样点

这一段不是“装饰”，而是 notebook 最强的地方：**把公式变成图，把图再变成直觉**。当你以后写别的波形、别的序列、别的信号处理代码时，这种“先生成、再比较、再可视化”的习惯会一直有用。


In [ ]:
from aurora.audio import sine
import aurora.aviz as aviz; aviz.style()
aviz.waveform(sine(440.0, duration=50/16000, sample_rate=16000), stem=True,
              title='440 Hz 正弦 — 前 50 个采样点');

In [ ]:
sr = 32
duration = 1.0
t = np.arange(int(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    zero_crossings = np.sum(np.diff(np.signbit(x)) != 0)
    print(f'freq={freq:>2}Hz | 前8点={np.round(x[:8], 2)} | 过零次数≈{zero_crossings}')


## 参数实验：Nyquist 频率的采样极限

把 `freq` 改成 `sample_rate/2`（Nyquist 频率），观察输出交替为 `[0, 1, 0, -1, ...]` 或 `[0, 0, 0, ...]`（取决于相位）——每个完整周期只剩两个采样点，正好够表示，但已是极限。这是 L34（混叠）的预习：当 `freq > sample_rate/2` 时，采样点不够，高频成分会被折叠成低频噪声。

In [ ]:
sr = 32
duration = 1.0
t = np.arange(int(duration * sr)) / sr

for freq in [1, 2, 4, 8, 16]:  # 16 = sr/2 = Nyquist 频率
    x = np.sin(2*np.pi*freq*t)
    print(f'freq={freq:>2}Hz | 前8点={np.round(x[:8], 2)} | max={x.max():.1f}')


## 本课收束

现在可以用 `my_sine(freq, duration, sample_rate)` 生成任意频率的正弦波。

`np.allclose(my_sine(...), reference_sine(...))` 返回 `True`，说明两个实现用的是同一套公式。

`aurora.audio.sine` 在 STFT 和 mel 特征提取的测试中生成标准信号，`my_sine` 是它的手动复现。

下一课（L34）会对这条正弦波降低采样率，观察高于 Nyquist 的频率如何被折叠到低频。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))


---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：正弦波手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：`freq=1 Hz, sample_rate=8 Hz, amplitude=1, duration=1s`
- N = ?（总点数）
- x[0] = sin(?) = ?
- x[2] = sin(?) = ?（应该是峰值）
- x[4] = sin(?) = ?（又回零？）

**问 2**：`freq=440 Hz, sample_rate=16000 Hz, duration=1s`
- N = ?
- x[0] = ?
- 完成一个周期需要多少个采样点？（`fₛ/f = ?`）

**问 3**：振幅 `A=0.5` 时，波形数值范围是 `[?, ?]`

**问 4**：为什么 `x[-1] ≠ 0`？（提示：最后一个采样点在 n=N−1，它的角度离 2π 的整数倍还差多少？）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：freq=1, sr=8, dur=1
freq, sr, dur = 1.0, 8, 1.0
N1 = round(dur * sr)
assert N1 == 8
n1 = np.arange(N1)
x1 = np.sin(2 * np.pi * freq * n1 / sr)
assert np.isclose(x1[0], 0.0, atol=1e-15)
assert np.isclose(x1[2], 1.0, atol=1e-15), f"x[2] 应=1（峰值），得到 {x1[2]:.6f}"
assert np.isclose(x1[4], 0.0, atol=1e-14), f"x[4] 应=0（过零），得到 {x1[4]:.2e}"
print(f"Q1 ✅  N={N1}，x={np.round(x1, 4)}")
print(f"      x[2]={x1[2]:.4f}（峰值），x[4]={x1[4]:.2e}（过零）")

# 验证 my_sine 一致性
try:
    mine = my_sine(freq, dur, sr)
    assert np.allclose(mine, x1, atol=1e-12)
    print(f"      my_sine 与手算结果吻合 ✅")
except NotImplementedError:
    print("⬜ Q1：请先实现 my_sine()，再运行对答案格")

# 问2：440 Hz, sr=16000
N2 = round(1.0 * 16000)
assert N2 == 16000
samples_per_cycle = 16000 / 440
assert np.isclose(samples_per_cycle, 36.36, atol=0.01)
print(f"Q2 ✅  N=16000，每周期 fₛ/f=16000/440≈{samples_per_cycle:.2f} 个采样点")

# 问3：amplitude=0.5
x3 = 0.5 * np.sin(2 * np.pi * 1.0 * np.arange(8) / 8)
assert np.max(x3) <= 0.5 + 1e-12 and np.min(x3) >= -0.5 - 1e-12
print(f"Q3 ✅  振幅=0.5，数值范围=[{np.min(x3):.4f}, {np.max(x3):.4f}]")

# 问4：非整数频率时 x[-1] 不回零
x4 = np.sin(2 * np.pi * 440 * np.arange(16000) / 16000)
last = x4[-1]
print(f"Q4 ✅  440Hz/16000Hz → x[-1]={last:.6f}≠0（最后一点在 n=N-1，比整周期终点早一个采样间隔）")
print(f"      x[-1] 的角度 = 2π·440·15999/16000 = 2π·440 − 2π·440/16000，不是 2π 的整数倍")
print("\n🎉 正弦波白板挑战通过！x[n]=A·sin(2πf·n/fₛ) 已内化。")

In [ ]:
# ✏️ 本课自评
l33_review = {
    "sine_formula":           None,  # 记住 x[n]=A·sin(2πf·n/fₛ)？True/False
    "my_sine_implemented":    None,  # my_sine 实现并通过 allclose 断言？True/False
    "samples_per_cycle":      None,  # 理解每周期采样数 = fₛ/f？True/False
    "nyquist_intuition":      None,  # 知道 f < fₛ/2 才能正常采样（Nyquist）？True/False
    "whiteboard_passed":      None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l33_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l33_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L33 全部通关！进入 L34：Nyquist 定理与混叠')

---

→ **下一课**　[L34 · Nyquist 定理与混叠](L34_aliasing.ipynb)

> 下节课将学习 **Nyquist 定理与混叠**：6kHz 正弦波被 8kHz 采样后会变成什么。